<div style="text-align: center;">
    <h1><strong>Alma Mater Studiorum - University of Bologna</strong></h1>
    

<div style="display:flex; justify-content:center; align-items:center; padding:5px;">
        <img src="../images_reports/image.png" style="height:300px; width:auto">
    </div>

<h2><strong>Cybersecurity</strong></h2>

<h3><strong>PROYECT</strong><br>
    <strong>Attacker Behavioral Profiling in SSH honeypots.</strong></h3>

<p><strong>STUDENTS</strong></p>
    <ul style="list-style-type:none; padding: 0;">
        <li><strong>Rubén Gil Martínez<strong></li>
        <li><strong>Guillermo López Pérez<strong></li>
        <li><strong>Jorge Mejías Donoso<strong></li>
    </ul>
</div>


- ### **Research question**

**Can machine learning models accurately classify attackers into distinct behavioral**
**profiles (automated bots, script kiddies, skilled operators) based on their command**
**sequences and interaction patterns in SSH honeypots?**


## **1) Data Pipeline for Cleaning and Preparation**

#### **STEP 1.1**: Examination of JSON files structure

In [1]:
import gzip
import json
import pandas as pd
from datetime import datetime
import os
from tqdm import tqdm

SAMPLE_GZ_PATH = "../DATASETS/raw_datasets/cyberlab_2019-11-09.json.gz"

In [4]:
def inspect_sample_json_gz(gz_path):
    print(f"[INFO] Reading: {gz_path}")
    
    # Load JSON from gzip
    with gzip.open(gz_path, 'rt', encoding='utf-8') as f:
        data = json.load(f)
    
    # Flatten the nested structure
    records = []
    for session_block in data:
        for session_id, events in session_block.items():
            for e in events:
                e["session_id"] = session_id
                records.append(e)
    
    # Convert to DataFrame
    df = pd.json_normalize(records, sep="/")
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")


    
    # --- BASIC INFORMATION ---
    print(f"\n[INFO] Total sessions: {df['session_id'].nunique():,}")
    print(f"[INFO] Total events:   {len(df):,}")
    print(f"[INFO] Time span of the day:       {df['timestamp'].min()} → {df['timestamp'].max()}")
    print(f"[INFO] Nº Columns:         {len(df.columns)}")
    print(df.columns.tolist())
    print("\n[INFO] Event type distribution:")
    print(df["eventid"].value_counts().head(20))
    
    print("\n[INFO] Example rows:")
    display(df.head(5))
    
    # --- OPTIONAL: Aggregates ---
    if "duration" in df.columns:
        print("\n[INFO] Duration statistics (seconds):")
        print(df["duration"].describe())
    
    if "geolocation_data/country_name" in df.columns:
        print("\n[INFO] Top 10 source countries:")
        print(df["geolocation_data/country_name"].value_counts().head(10))
    
    if "username" in df.columns:
        print("\n[INFO] Top 10 usernames used:")
        print(df["username"].dropna().value_counts().head(10))
    
    return df

# --- RUN ANALYSIS ---
sample_df = inspect_sample_json_gz(SAMPLE_GZ_PATH)


[INFO] Reading: ../DATASETS/raw_datasets/cyberlab_2019-11-09.json.gz

[INFO] Total sessions: 78,685
[INFO] Total events:   531,496
[INFO] Time span of the day:       2019-11-09 00:00:00.402474+00:00 → 2019-11-10 00:02:57.619028+00:00
[INFO] Nº Columns:         50
['session_id', 'dst_host_identifier', 'eventid', 'timestamp', 'src_ip_identifier', 'dst_ip_identifier', 'message', 'protocol', 'src_port', 'sensor', 'arch', 'duration', 'ssh_client_version', 'username', 'password', 'hasshAlgorithms', 'macCS', 'langCS', 'compCS', 'encCS', 'hassh', 'kexAlgs', 'keyAlgs', 'fingerprint', 'key', 'type', 'outfile', 'destfile', 'duplicate', 'shasum', 'url', 'ttylog', 'size', 'filename', 'data', 'geolocation_data/country_name', 'geolocation_data/longitude', 'geolocation_data/continent_code', 'geolocation_data/country_code3', 'geolocation_data/country_code2', 'geolocation_data/ip', 'geolocation_data/latitude', 'geolocation_data/timezone', 'geolocation_data/location/lat', 'geolocation_data/location/lon',

,session_id,dst_host_identifier,eventid,timestamp,src_ip_identifier,dst_ip_identifier,message,protocol,src_port,sensor,...,geolocation_data/ip,geolocation_data/latitude,geolocation_data/timezone,geolocation_data/location/lat,geolocation_data/location/lon,geolocation_data/region_name,geolocation_data/region_code,geolocation_data/city_name,geolocation_data/postal_code,geolocation_data/dma_code
0,7a440a2b2fec,38a92c83b3a3a04710d0470d06f379e61300d2dbfe31f1...,cowrie.session.connect,2019-11-09 00:03:37.032073+00:00,358efd321aefa5984b484c640c71359c82b7025d47a53e...,None,New connection: 358efd321aefa5984b484c640c7135...,ssh,33698.0,ubuntu_basic_pool,...,358efd321aefa5984b484c640c71359c82b7025d47a53e...,13.75,Asia/Bangkok,13.75,100.4667,NaN,NaN,NaN,NaN,NaN
1,7a440a2b2fec,38a92c83b3a3a04710d0470d06f379e61300d2dbfe31f1...,cowrie.client.version,2019-11-09 00:03:37.284685+00:00,358efd321aefa5984b484c640c71359c82b7025d47a53e...,None,Remote SSH version: b'SSH-2.0-libssh-0.6.3',None,NaN,ubuntu_basic_pool,...,358efd321aefa5984b484c640c71359c82b7025d47a53e...,13.75,Asia/Bangkok,13.75,100.4667,NaN,NaN,NaN,NaN,NaN
2,7a440a2b2fec,38a92c83b3a3a04710d0470d06f379e61300d2dbfe31f1...,cowrie.client.kex,2019-11-09 00:03:37.534477+00:00,358efd321aefa5984b484c640c71359c82b7025d47a53e...,None,SSH client hassh fingerprint: 51cba57125523ce4...,None,NaN,ubuntu_basic_pool,...,358efd321aefa5984b484c640c71359c82b7025d47a53e...,13.75,Asia/Bangkok,13.75,100.4667,NaN,NaN,NaN,NaN,NaN
3,7a440a2b2fec,38a92c83b3a3a04710d0470d06f379e61300d2dbfe31f1...,cowrie.login.failed,2019-11-09 00:03:38.620092+00:00,358efd321aefa5984b484c640c71359c82b7025d47a53e...,None,login attempt [ten*Inko/root] failed,None,NaN,ubuntu_basic_pool,...,358efd321aefa5984b484c640c71359c82b7025d47a53e...,13.75,Asia/Bangkok,13.75,100.4667,NaN,NaN,NaN,NaN,NaN
4,7a440a2b2fec,38a92c83b3a3a04710d0470d06f379e61300d2dbfe31f1...,cowrie.session.closed,2019-11-09 00:03:39.873425+00:00,358efd321aefa5984b484c640c71359c82b7025d47a53e...,None,Connection lost after 2 seconds,None,NaN,ubuntu_basic_pool,...,358efd321aefa5984b484c640c71359c82b7025d47a53e...,13.75,Asia/Bangkok,13.75,100.4667,NaN,NaN,NaN,NaN,NaN



[INFO] Duration statistics (seconds):
count    109232.000000
mean         18.911747
std          47.707123
min           0.000457
25%           0.207092
50%           0.655274
75%           3.923330
max         698.199176
Name: duration, dtype: float64

[INFO] Top 10 source countries:
geolocation_data/country_name
China                166811
Ireland               68622
Russia                48208
United States         33482
France                20578
United Kingdom        12633
Republic of Korea     11511
Netherlands            9615
India                  7301
Singapore              6983
Name: count, dtype: int64

[INFO] Top 10 usernames used:
username
root      80018
admin      3341
nproc      1675
adm         304
123456      218
1234        187
test        165
111111      157
123         143
user        140
Name: count, dtype: int64


# CyberLab Honeynet Dataset Structure

The CyberLab Honeynet dataset is composed of daily `.json.gz` files containing SSH and Telnet interaction logs collected by Cowrie honeypots. Each file represents one day of captured activity and includes all attacker sessions initiated during that period.

## Hierarchical Organization

- **File level:**  
  Each file corresponds to a single day of data collected from multiple honeypot sensors.

- **Session level:**  
  Each element in the top-level list represents one **attack session**, identified by a unique `session_id`.  
  A session includes all events generated by one attacker connection to the honeypot.

- **Event level:**  
  Within each session, there is a list of **event dictionaries**.  
  Each event records a specific action or system response such as:
  - Connection initiation (`cowrie.session.connect`)
  - Login attempt (`cowrie.login.success` / `cowrie.login.failed`)
  - Command execution (`cowrie.command.input`)
  - Session termination (`cowrie.session.closed`)


#### **STEP 1.2**: Cleaning Guidelines

1) Flatten nested fields (already done by json_normalize, using / separator).

2) Ensure type casting according to this schema — categorical for limited-value fields, float for coordinates/durations, string for identifiers.

3) Drop highly sparse fields (>90% NaN) unless strategically relevant.

4) Normalize lists like macCS, encCS, kexAlgs, and keyAlgs into string representations (comma-separated).

5) Deduplicate by session_id + timestamp + eventid if necessary.

6) Store the cleaned version as Parquet with preserved schema.

In [ ]:
DATA_DIR = "../raw_datasets/"  
OUTPUT_DIR = "../cleaned_parquet_datasets/"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def load_and_flatten_json_gz(filepath):
    """Load and flatten one .json.gz file."""
    print(f"\n[INFO] Processing file: {filepath}")
    records = []
    try:
        with gzip.open(filepath, 'rt', encoding='utf-8') as f:
            # Leer el archivo línea por línea para evitar cargar todo en memoria
            data = json.load(f)
            total_blocks = len(data)
            print(f"[INFO] Found {total_blocks} session blocks")
            
            for i, session_block in enumerate(data, 1):
                if i % 100 == 0:
                    print(f"[INFO] Processing block {i}/{total_blocks}")
                for session_id, events in session_block.items():
                    for e in events:
                        e["session_id"] = session_id
                        records.append(e)
    except Exception as e:
        print(f"[ERROR] Error reading file: {str(e)}")
        raise
    print(f"[INFO] Total records found: {len(records)}")
    return pd.json_normalize(records, sep="/", errors='ignore')



def flatten_list_fields(df):
    """
    Detect and flatten list-type fields such as algorithm sets.
    Converts list columns into comma-separated strings.
    """
    list_fields = ["macCS", "encCS", "kexAlgs", "keyAlgs"]
    print("[INFO] Flattening list fields...")
    for field in list_fields:
        if field in df.columns:
            df[field] = df[field].apply(
                lambda x: ", ".join(x) if isinstance(x, list) else x
            )
    return df



def clean_dataframe(df):
    """Basic cleaning and normalization."""

    print("[INFO] Cleaning dataframe...")
    print(f"[INFO] Initial shape: {df.shape}")
    
    # Ensure valid timestamps
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df[df["timestamp"].notna()]
    print(f"[INFO] Shape after timestamp cleaning: {df.shape}")
    
    # Drop extremely sparse columns (>90% NaN)
    null_ratio = df.isna().mean()
    sparse_cols = null_ratio[null_ratio >= 0.90].index
    print(f"[INFO] Dropping {len(sparse_cols)} sparse columns")
    df = df[null_ratio[null_ratio < 0.90].index]
    
    # Normalize datatypes for certain columns
    for col in ["eventid", "protocol", "sensor", "arch"]:
        if col in df.columns:
            df[col] = df[col].astype("category")
    
    print(f"[INFO] Final shape: {df.shape}")
    return df





def process_all_files(data_dir=DATA_DIR, output_dir=OUTPUT_DIR):
    files = sorted([f for f in os.listdir(data_dir) if f.endswith(".json.gz")])
    total_files = len(files)
    print(f"[INFO] Found {total_files} JSON.GZ files.")
    
    for i, file in enumerate(files, 1):
        path = os.path.join(data_dir, file)
        print(f"\n[INFO] Processing file {i}/{total_files}: {file}")
        try:
            df = load_and_flatten_json_gz(path)
            df = flatten_list_fields(df)
            df = clean_dataframe(df)
            
            if len(df) == 0:
                print("[WARNING] Empty dataframe after processing, skipping...")
                continue
                
            base = os.path.splitext(os.path.splitext(file)[0])[0]
            output_path = os.path.join(output_dir, f"{base}.parquet")
            df.to_parquet(output_path, index=False)
            print(f"[SUCCESS] Saved to {output_path}")
            
        except Exception as e:
            print(f"[ERROR] Failed to process {file}: {str(e)}")
            continue

# Ejecutar el procesamiento
print("[INFO] Starting processing pipeline...")
process_all_files()

[INFO] Starting processing pipeline...
[INFO] Found 113 JSON.GZ files.

[INFO] Processing file 1/113: cyberlab_2019-11-09.json.gz

[INFO] Processing file: ../datasets/cyberlab_2019-11-09.json.gz
[INFO] Found 78685 session blocks
[INFO] Processing block 100/78685
[INFO] Processing block 200/78685
[INFO] Processing block 300/78685
[INFO] Processing block 400/78685
[INFO] Processing block 500/78685
[INFO] Processing block 600/78685
[INFO] Processing block 700/78685
[INFO] Processing block 800/78685
[INFO] Processing block 900/78685
[INFO] Processing block 1000/78685
[INFO] Processing block 1100/78685
[INFO] Processing block 1200/78685
[INFO] Processing block 1300/78685
[INFO] Processing block 1400/78685
[INFO] Processing block 1500/78685
[INFO] Processing block 1600/78685
[INFO] Processing block 1700/78685
[INFO] Processing block 1800/78685
[INFO] Processing block 1900/78685
[INFO] Processing block 2000/78685
[INFO] Processing block 2100/78685
[INFO] Processing block 2200/78685
[INFO] Pro

#### **STEP 1.3**: Efficient loading using Dask

In [16]:
from dask import config
import multiprocessing

print("[INFO] Logical CPU cores available:", multiprocessing.cpu_count())
print("[INFO] Dask scheduler:", config.get('scheduler', 'threads'))


[INFO] Logical CPU cores available: 16
[INFO] Dask scheduler: threads


- Taking advantage from the parallel computation, *Dask* provides us the tools to do a more efficient features engineering.

In [17]:
config.set(scheduler='threads', num_workers=8)  

- Due to the substantial size of the dataset (10.5 GB compressed), we decided to select a small yet sufficiently representative subset in order to address the various characteristics and behaviors exhibited by the attackers.

In [18]:
import os
import dask.dataframe as dd

# --- 1. Define the cleaned dataset path ---
DATA_DIR = "../DATASETS/cleaned_parquet_datasets"

# --- 2. List and sort Parquet files ---
parquet_files = sorted([os.path.join(DATA_DIR, f) 
                        for f in os.listdir(DATA_DIR) if f.endswith(".parquet")])

# --- 3. Take a subset due to the enormous size of the real dataset ---
sample_files = parquet_files[0:10]
print("\n[INFO] Loading files:")
for f in sample_files:
    print(f"    {os.path.basename(f)}")

# --- 4. Read them with Dask (lazy, parallel) ---
ddf_list = [dd.read_parquet(f, engine="pyarrow") for f in sample_files]

# --- 5. Merge them (schema union) ---
print("\n[INFO] Concatenating Dask DataFrames...")
ddf = dd.concat(ddf_list, join="outer", axis=0, interleave_partitions=True)

# --- 6. Show structure ---
print("\n[INFO] Preview of unified schema:")
print(ddf.dtypes.head(10))  # Show first few column dtypes



[INFO] Loading files:
    cyberlab_2019-11-09.parquet
    cyberlab_2019-11-10.parquet
    cyberlab_2019-11-11.parquet
    cyberlab_2019-11-12.parquet
    cyberlab_2019-11-13.parquet
    cyberlab_2019-11-14.parquet
    cyberlab_2019-11-15.parquet
    cyberlab_2019-11-16.parquet
    cyberlab_2019-11-17.parquet
    cyberlab_2019-11-18.parquet

[INFO] Concatenating Dask DataFrames...

[INFO] Preview of unified schema:
session_id                          object
dst_host_identifier                 object
eventid                           category
timestamp              datetime64[ns, UTC]
src_ip_identifier                   object
message                             object
protocol                          category
src_port                           float64
sensor                            category
duration                           float64
dtype: object


In [19]:
# 1. Seleccionas tus dos columnas de interés del Dask DataFrame (ddf)
columnas_interes = ddf[ddf['eventid'] == "cowrie.command.input"][['session_id', 'message']]

# 2. Computas (traes a RAM) y guardas en una sola línea
# index=False evita que se guarde el número de fila (0, 1, 2...) en el CSV
columnas_interes.compute().to_csv('../DATASETS/executed_commands.csv', index=False)

print("Exportación completada. Archivo guardado como 'dataset_columnas_unificadas.csv'")

Exportación completada. Archivo guardado como 'dataset_columnas_unificadas.csv'


In [11]:
print("[INFO] Sorting by session_id...")
ddf = ddf.map_partitions(lambda df: df.sort_values(["session_id", "timestamp"]))


def extract_session_features(df):
    """Extract per-session temporal, behavioral, and command-based features."""
    grouped = df.groupby("session_id")

    def compute_features(g):
        g = g.sort_values("timestamp")


        # --- TEMPORAL FEATURES --- 
        num_events = len(g)
        num_commands = g[g["eventid"].str.contains("cowrie.command.input", na=False)].shape[0]
        session_duration = (g["timestamp"].max() - g["timestamp"].min()).total_seconds()
        inter_times = g[g["eventid"].str.contains("cowrie.command.input", na=False)]["timestamp"].diff().dt.total_seconds().dropna()
        max_inter = inter_times.max() if not inter_times.empty else 0
        mean_inter = inter_times.mean() if not inter_times.empty else 0
        std_inter = inter_times.std() if not inter_times.empty else 0


        # --- BEHAVIORAL FEATURES --- 
        event_counts = g["eventid"].value_counts().to_dict()
        num_failures = event_counts.get("cowrie.command.failed", 0)
        num_downloads = event_counts.get("cowrie.session.file_download", 0)
        num_uploads = event_counts.get("cowrie.session.file_upload", 0)
        num_recon = (
            event_counts.get("cowrie.login.failed", 0) 
            + event_counts.get("cowrie.client.version", 0)
            + event_counts.get("cowrie.client.kex", 0)
            + event_counts.get("cowrie.direct-tcpip.request", 0)
            + event_counts.get("cowrie.client.fingerprint", 0)
        )
        num_exploit = (
            event_counts.get("cowrie.login.success", 0)
            + event_counts.get("cowrie.direct-tcpip.data", 0)
            + num_downloads
            + num_uploads
        )
        command_error_rate = num_failures / (num_commands + 1e-6)
        login_error_rate = event_counts.get("cowrie.login.failed", 0) / (event_counts.get("cowrie.login.success", 0) + event_counts.get("cowrie.login.failed", 0) + 1e-6)
        file_transfer_ratio = (num_downloads + num_uploads) / (num_commands + 1e-6)
        reconnaissance_ratio = num_recon / (num_events + 1e-6)
        exploit_ratio = num_exploit / (num_events + 1e-6)




        # --- COMMAND-BASED FEATURES ---
        cmd_events = g[g["eventid"].str.contains("cowrie.command.input", na=False)] # Filter the events by only those which are real executed commands
        
        unique_commands_ratio = 0
        command_diversity = 0
        
        if not cmd_events.empty:
            cmds = cmd_events["message"].dropna().astype(str).str.strip()

            if not cmds.empty:
                cleaned_cmds = cmds.str.replace('CMD: ', '', n=1, regex=False)
                piped_cmds = cleaned_cmds.str.split('|').explode()
                base_cmds = piped_cmds.str.strip().str.split(r'\s+', n=1).str[0].str.lower() 
                base_cmds = base_cmds[base_cmds.notna() & (base_cmds != '')] 
                total_cmds = len(base_cmds)

                if total_cmds > 0:
                    cmd_counts = base_cmds.value_counts()
                    command_diversity = len(cmd_counts)
                    unique_commands_ratio = 1 - ((command_diversity - 1) / total_cmds) 
                else:
                    command_diversity = 0
                    unique_commands_ratio = 0
        


        return pd.Series({

            # --- TEMPORAL ---
            "num_events": num_events,
            "num_commands": num_commands,
            "session_duration": session_duration,
            "max_inter_command_time": max_inter,
            "mean_inter_command_time": mean_inter,
            "std_inter_command_time": std_inter,


            # --- BEHAVIORAL ---
            "command_error_rate": command_error_rate,
            "file_transfer_ratio": file_transfer_ratio,
            "login_error_rate": login_error_rate,
            "reconnaissance_ratio": reconnaissance_ratio,
            "exploit_ratio": exploit_ratio,


            # --- COMMAND-BASED ---
            "unique_commands_ratio": unique_commands_ratio,
            "command_diversity": command_diversity
        })

    features = grouped.apply(compute_features)
    return features


print("[INFO] Extracting per-session event-based features...")
session_features = ddf.map_partitions(extract_session_features).compute(num_workers=8)

print("[SUCCESS] Event-based feature extraction completed!")
final_dataset = session_features[session_features['num_commands'] > 0]


[INFO] Sorting by session_id...
[INFO] Extracting per-session event-based features...


C:\Users\ruben\AppData\Local\Temp\ipykernel_59756\3730412200.py:100: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  features = grouped.apply(compute_features)
C:\Users\ruben\AppData\Local\Temp\ipykernel_59756\3730412200.py:100: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  features = grouped.apply(compute_features)
C:\Users\ruben\AppData\Local\Temp\ipykernel_59756\3730412200.py:100: FutureWarning: DataFrameGroupBy.app

[SUCCESS] Event-based feature extraction completed!


#### **STEP 1.4**: Store the final dataset in CSV format

In [12]:
final_dataset

,num_events,num_commands,session_duration,max_inter_command_time,mean_inter_command_time,std_inter_command_time,command_error_rate,file_transfer_ratio,login_error_rate,reconnaissance_ratio,exploit_ratio,unique_commands_ratio,command_diversity
session_id,,,,,,,,,,,,,
00081b571122,78.0,23.0,31.249671,6.333993,1.160163,1.173813,0.086957,0.173913,0.0,0.025641,0.064103,0.538462,19.0
0076d693f7fd,78.0,23.0,32.132288,6.536506,1.194688,1.211933,0.086957,0.173913,0.0,0.025641,0.064103,0.538462,19.0
00aa3ae5d33a,78.0,23.0,36.995369,7.981335,1.388428,1.529704,0.086957,0.173913,0.0,0.025641,0.064103,0.538462,19.0
010ad96c3f21,16.0,7.0,68.448152,19.502649,3.257964,7.958238,0.428571,0.000000,0.0,0.000000,0.062500,0.285714,6.0
012f382592c2,78.0,23.0,6.000740,0.534131,0.201693,0.094307,0.086957,0.173913,0.0,0.025641,0.064103,0.538462,19.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ff1982e219ea,41.0,19.0,29.589802,1.698201,1.208509,0.208796,0.000000,0.000000,0.0,0.048780,0.024390,0.558824,16.0
ff30a7409bb0,78.0,23.0,26.672534,5.367225,0.980449,0.995405,0.086957,0.173913,0.0,0.025641,0.064103,0.538462,19.0
ff46f283de84,18.0,8.0,2.415998,0.274037,0.073823,0.119185,0.375000,0.000000,0.0,0.000000,0.055556,0.300000,8.0


In [13]:
final_dataset.to_csv("../DATASETS/attacker_behavioral_profiles_dataset_3.0.csv", index=True)

- With the dataset ready, we can proceed to Part 2 of the project, where we will use unsupervised algorithms to model the behavioral space of the attackers. This will enable us to accurately classify new attackers that interact with our SSH honeypots.